In [2]:
from pathlib import Path
import csv
import random
import shutil
from collections import defaultdict

UNIFIED = Path(r"F:\Datasets\VisioDECT_YOLO_Unified")
OUTPUT = Path(r"F:\Datasets\VisioDECT_YOLO_GroupSplit")

IMAGES_ALL = UNIFIED / "images" / "all"
LABELS_ALL = UNIFIED / "labels" / "all"
MANIFEST_PATH = UNIFIED / "manifest.csv"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Unified exists:", UNIFIED.exists())
print("Images folder exists:", IMAGES_ALL.exists())
print("Labels folder exists:", LABELS_ALL.exists())
print("Manifest exists:", MANIFEST_PATH.exists())

Unified exists: True
Images folder exists: True
Labels folder exists: True
Manifest exists: True


In [ ]:
SEED = 42


FRAME_STEP = 5

# 60% train 20% val 20% test
VAL_GROUPS_PER_SCENARIO = 1
TEST_GROUPS_PER_SCENARIO = 1

random.seed(SEED)

print("Seed:", SEED)
print("Frame step:", FRAME_STEP)

Seed: 42
Frame step: 5


In [4]:
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

print("Manifest rows:", len(manifest_rows))

print("Columns:")
print(manifest_rows[0].keys())

print("\nFirst row:")
print(manifest_rows[0])

Manifest rows: 20617
Columns:
dict_keys(['new_image', 'new_label', 'original_image', 'model', 'scenario', 'group', 'label_source', 'num_boxes'])

First row:
{'new_image': 'Anafi-Extended_Cloudy_Anafi_Extended_Cloudy_(1).jpg', 'new_label': 'Anafi-Extended_Cloudy_Anafi_Extended_Cloudy_(1).txt', 'original_image': 'Anafi_Extended_Cloudy (1).jpg', 'model': 'Anafi-Extended', 'scenario': 'Cloudy', 'group': 'Anafi-Extended/Cloudy', 'label_source': 'F:\\Datasets\\VisioDECT Scenario-Based Multi-Drone Detection\\VisioDECT Dataset Upload\\Anafi-Extended\\labels\\cloudy\\csv.csv', 'num_boxes': '1'}


In [5]:
groups = {}

for row in manifest_rows:
    group = row["group"]
    model = row["model"]
    scenario = row["scenario"]
    groups[group] = {
        "model": model,
        "scenario": scenario
    }

print("Number of groups:", len(groups))

for group in sorted(groups.keys()):
    print(group)

Number of groups: 18
Anafi-Extended/Cloudy
Anafi-Extended/Evening
Anafi-Extended/Sunny
DJIFPV/Cloudy
DJIFPV/Evening
DJIFPV/Sunny
DJIPhantom/Cloudy
DJIPhantom/Evening
DJIPhantom/Sunny
EFT-E410S/Cloudy
EFT-E410S/Evening
EFT-E410S/Sunny
Mavic_Air/Cloudy
Mavic_Air/Evening
Mavic_Air/Sunny
Mavic_Enterprise/Cloudy
Mavic_Enterprise/Evening
Mavic_Enterprise/Sunny


In [6]:
group_counts = defaultdict(int)

for row in manifest_rows:
    group_counts[row["group"]] += 1

print("Images per group:")

for group in sorted(group_counts.keys()):
    print(f"{group}: {group_counts[group]}")

Images per group:
Anafi-Extended/Cloudy: 1200
Anafi-Extended/Evening: 1200
Anafi-Extended/Sunny: 989
DJIFPV/Cloudy: 1112
DJIFPV/Evening: 1127
DJIFPV/Sunny: 1200
DJIPhantom/Cloudy: 900
DJIPhantom/Evening: 1200
DJIPhantom/Sunny: 1200
EFT-E410S/Cloudy: 1200
EFT-E410S/Evening: 1152
EFT-E410S/Sunny: 1200
Mavic_Air/Cloudy: 1165
Mavic_Air/Evening: 1163
Mavic_Air/Sunny: 1191
Mavic_Enterprise/Cloudy: 1188
Mavic_Enterprise/Evening: 1098
Mavic_Enterprise/Sunny: 1132


In [7]:
manifest_rows_sorted = sorted(
    manifest_rows,
    key=lambda r: (r["group"], r["new_image"])
)

rows_by_group = defaultdict(list)

for row in manifest_rows_sorted:
    rows_by_group[row["group"]].append(row)

sampled_rows = []

for group, rows in rows_by_group.items():
    for idx, row in enumerate(rows):
        if idx % FRAME_STEP == 0:
            sampled_rows.append(row)

print("Original rows:", len(manifest_rows))
print("Rows after FRAME_STEP sampling:", len(sampled_rows))

sampled_group_counts = defaultdict(int)

for row in sampled_rows:
    sampled_group_counts[row["group"]] += 1

print("\nSampled images per group:")

for group in sorted(sampled_group_counts.keys()):
    print(f"{group}: {sampled_group_counts[group]}")

Original rows: 20617
Rows after FRAME_STEP sampling: 4128

Sampled images per group:
Anafi-Extended/Cloudy: 240
Anafi-Extended/Evening: 240
Anafi-Extended/Sunny: 198
DJIFPV/Cloudy: 223
DJIFPV/Evening: 226
DJIFPV/Sunny: 240
DJIPhantom/Cloudy: 180
DJIPhantom/Evening: 240
DJIPhantom/Sunny: 240
EFT-E410S/Cloudy: 240
EFT-E410S/Evening: 231
EFT-E410S/Sunny: 240
Mavic_Air/Cloudy: 233
Mavic_Air/Evening: 233
Mavic_Air/Sunny: 239
Mavic_Enterprise/Cloudy: 238
Mavic_Enterprise/Evening: 220
Mavic_Enterprise/Sunny: 227


In [9]:
# ============================================================
# MODEL-LEVEL SPLIT
# ============================================================
# Goal:
# Each drone model should appear in only one split.
#
# This prevents:
# Anafi-Extended in train and val
# DJIFPV in train and val
#
# Split logic:
# 6 models total:
# - 4 models train
# - 1 model val
# - 1 model test
# ============================================================

models = sorted(set(row["model"] for row in sampled_rows))

print("Models found:")
for m in models:
    print("-", m)

random.seed(SEED)
shuffled_models = models.copy()
random.shuffle(shuffled_models)

test_models = shuffled_models[:1]
val_models = shuffled_models[1:2]
train_models = shuffled_models[2:]

split_by_model = {}

for model in train_models:
    split_by_model[model] = "train"

for model in val_models:
    split_by_model[model] = "val"

for model in test_models:
    split_by_model[model] = "test"

print("\nModel split:")
for model in sorted(split_by_model.keys()):
    print(f"{model}: {split_by_model[model]}")

Models found:
- Anafi-Extended
- DJIFPV
- DJIPhantom
- EFT-E410S
- Mavic_Air
- Mavic_Enterprise

Model split:
Anafi-Extended: train
DJIFPV: val
DJIPhantom: train
EFT-E410S: test
Mavic_Air: train
Mavic_Enterprise: train


In [10]:
split_rows = []

for row in sampled_rows:
    row_copy = row.copy()
    row_copy["split"] = split_by_model[row["model"]]
    split_rows.append(row_copy)

split_counts = defaultdict(int)

for row in split_rows:
    split_counts[row["split"]] += 1

print("Images per split:")

for split in ["train", "val", "test"]:
    print(split, split_counts[split])

Images per split:
train 2728
val 689
test 711


In [11]:
models_per_split = defaultdict(set)
scenarios_per_split = defaultdict(set)
groups_per_split = defaultdict(set)

for row in split_rows:
    split = row["split"]
    models_per_split[split].add(row["model"])
    scenarios_per_split[split].add(row["scenario"])
    groups_per_split[split].add(row["group"])

print("=" * 80)
print("MODELS PER SPLIT")
print("=" * 80)

for split in ["train", "val", "test"]:
    print()
    print(split.upper())
    print("Models:", sorted(models_per_split[split]))
    print("Scenarios:", sorted(scenarios_per_split[split]))
    print("Number of groups:", len(groups_per_split[split]))
    for group in sorted(groups_per_split[split]):
        print("-", group)

# Leakage check
train_models = models_per_split["train"]
val_models = models_per_split["val"]
test_models = models_per_split["test"]

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print("Train ∩ Val:", train_models & val_models)
print("Train ∩ Test:", train_models & test_models)
print("Val ∩ Test:", val_models & test_models)

if not (train_models & val_models) and not (train_models & test_models) and not (val_models & test_models):
    print("\n No model leakage. Each drone model appears in only one split.")
else:
    print("\n Model leakage found.")

MODELS PER SPLIT

TRAIN
Models: ['Anafi-Extended', 'DJIPhantom', 'Mavic_Air', 'Mavic_Enterprise']
Scenarios: ['Cloudy', 'Evening', 'Sunny']
Number of groups: 12
- Anafi-Extended/Cloudy
- Anafi-Extended/Evening
- Anafi-Extended/Sunny
- DJIPhantom/Cloudy
- DJIPhantom/Evening
- DJIPhantom/Sunny
- Mavic_Air/Cloudy
- Mavic_Air/Evening
- Mavic_Air/Sunny
- Mavic_Enterprise/Cloudy
- Mavic_Enterprise/Evening
- Mavic_Enterprise/Sunny

VAL
Models: ['DJIFPV']
Scenarios: ['Cloudy', 'Evening', 'Sunny']
Number of groups: 3
- DJIFPV/Cloudy
- DJIFPV/Evening
- DJIFPV/Sunny

TEST
Models: ['EFT-E410S']
Scenarios: ['Cloudy', 'Evening', 'Sunny']
Number of groups: 3
- EFT-E410S/Cloudy
- EFT-E410S/Evening
- EFT-E410S/Sunny

LEAKAGE CHECK
Train ∩ Val: set()
Train ∩ Test: set()
Val ∩ Test: set()

 No model leakage. Each drone model appears in only one split.


In [12]:
CLEAN_OUTPUT = True

if OUTPUT.exists() and CLEAN_OUTPUT:
    shutil.rmtree(OUTPUT)
    print("Removed old split output folder.")

for split in ["train", "val", "test"]:
    (OUTPUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Output folder ready:")
print(OUTPUT)

Output folder ready:
F:\Datasets\VisioDECT_YOLO_GroupSplit


In [13]:
missing_images = []
missing_labels = []

for row in split_rows:
    split = row["split"]

    src_img = IMAGES_ALL / row["new_image"]
    src_lbl = LABELS_ALL / row["new_label"]

    dst_img = OUTPUT / "images" / split / row["new_image"]
    dst_lbl = OUTPUT / "labels" / split / row["new_label"]

    if not src_img.exists():
        missing_images.append(str(src_img))
        continue

    if not src_lbl.exists():
        missing_labels.append(str(src_lbl))
        continue

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_lbl, dst_lbl)

print("Copy finished.")
print("Missing images:", len(missing_images))
print("Missing labels:", len(missing_labels))

Copy finished.
Missing images: 0
Missing labels: 0


In [14]:
split_manifest_path = OUTPUT / "split_manifest.csv"
split_groups_path = OUTPUT / "split_groups.csv"
split_models_path = OUTPUT / "split_models.csv"

fieldnames = list(split_rows[0].keys())

with open(split_manifest_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(split_rows)

# Save group-level info
group_rows = []

seen_groups = {}

for row in split_rows:
    group = row["group"]
    if group not in seen_groups:
        seen_groups[group] = {
            "group": group,
            "model": row["model"],
            "scenario": row["scenario"],
            "split": row["split"]
        }

group_rows = list(seen_groups.values())

with open(split_groups_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["group", "model", "scenario", "split"]
    )
    writer.writeheader()
    writer.writerows(group_rows)

# Save model-level info
model_rows = []

for model, split in split_by_model.items():
    model_rows.append({
        "model": model,
        "split": split
    })

with open(split_models_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["model", "split"]
    )
    writer.writeheader()
    writer.writerows(model_rows)

print("Saved:")
print(split_manifest_path)
print(split_groups_path)
print(split_models_path)

Saved:
F:\Datasets\VisioDECT_YOLO_GroupSplit\split_manifest.csv
F:\Datasets\VisioDECT_YOLO_GroupSplit\split_groups.csv
F:\Datasets\VisioDECT_YOLO_GroupSplit\split_models.csv


In [15]:
data_yaml_path = OUTPUT / "data.yaml"

with open(data_yaml_path, "w", encoding="utf-8") as f:
    f.write(f"path: {str(OUTPUT).replace(chr(92), '/')}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("test: images/test\n")
    f.write("\n")
    f.write("names:\n")
    f.write("  0: drone\n")

print("Created data.yaml:")
print(data_yaml_path)

print("\nContent:")
print(open(data_yaml_path, "r", encoding="utf-8").read())

Created data.yaml:
F:\Datasets\VisioDECT_YOLO_GroupSplit\data.yaml

Content:
path: F:/Datasets/VisioDECT_YOLO_GroupSplit
train: images/train
val: images/val
test: images/test

names:
  0: drone



In [16]:
for split in ["train", "val", "test"]:
    images = [
        p for p in (OUTPUT / "images" / split).iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    labels = [
        p for p in (OUTPUT / "labels" / split).iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    ]

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    print("=" * 60)
    print(split.upper())
    print("=" * 60)
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Images without labels:", len(image_stems - label_stems))
    print("Labels without images:", len(label_stems - image_stems))

TRAIN
Images: 2728
Labels: 2728
Images without labels: 0
Labels without images: 0
VAL
Images: 689
Labels: 689
Images without labels: 0
Labels without images: 0
TEST
Images: 711
Labels: 711
Images without labels: 0
Labels without images: 0
